# Giải thích:
Đây là cách để tạo ra file sách so sánh (compare book)
Gồm 2 sách: left-book và right-book

Ghi chú: mối sách số file cần so sánh phải như nhau, và có bản map 1-1 giữa sách bên trái và sách bên phải
ví du:
```
left         right
chap1.md     chuong-1.md   
chap2.md     chuong-2.md
```

1. tạo filelist.js cho từng book left+right (3make_mucluc.ipynb)
Kết quả tạo ra file `mucluc.md` và `filelist.js` trong 
- mucluc.md dùng để làm file index hoặc bất cứ thứ gì
- filelist.js 
danh sách các text và link dùng để làm button [next][back] xem `config.ts` code phần `BOOK_NAV`
`{ text: " MN 1. CÁC NGUYÊN LÝ", link: "/kinhtrungbo/pali-vi/mn-001.md" },`


2. Tạo ra sách so sánh (l-r) 
 1. copy/paste `filelist.js` ở bước 1 vào code ở dưới (left) 
 2. copy/paste `filelist.js` ở bước 1 vào code ở dưới (right) 
 (sẽ lấy text-url theo tài liệu của right)

In [1]:
import os
import json
from slugify import slugify
import re

def merge_lists(list_left, list_right, count, left_title, right_title, output_dir, output_val, url_path, pre_title='', pre_slug='', title_kind="right"):
    """
    Merges two lists of dictionaries and creates a new list with combined data.

    Args:
        list_a: The first list of dictionaries.
        list_b: The second list of dictionaries.
        count: The number of elements to merge.
        left_title:  The title for the 'left' side (from list_b).
        right_title: The title for the 'right' side (from list_a).
        dest_folder: save mucluc.md path
        pre_slug: dùng dnc='kinh trường bộ compare'

    Returns:
        A new list of dictionaries with the merged data.
    """

    merged_list = []
    list_nav = list_left if title_kind == "left" else list_right

    # def slug_link(item):
    #     return f"{pre_slug}{os.path.basename(item['link'])}".replace('.md', '')[2:]

    # def make_slug(text: str) -> str:
    #     match = re.match(r'^[A-Z]+\s+(\d+)\.\s+(.+)$', text.strip())
    #     if not match:
    #         raise ValueError(f"Không đúng định dạng: {text!r}")

    #     num   = int(match.group(1))          # 1  →  001
    #     title = match.group(2)               # "KINH LƯỚI TRỜI"

    #     return f"{pre_slug}{num:03d}-{slugify(title)}"

    for i in range(min(count, len(list_left), len(list_right))):  # Iterate up to the shortest list length or 'count'
        left_item = list_left[i]
        right_item = list_right[i]

        # slug = make_slug(title_item[i]['text'])
        slug = f"{pre_slug}{os.path.basename(list_nav[i]['link']).replace('.md', '')[3:]}"

        title = f"{pre_title}{list_nav[i]['text']}"


        # 3. & 4. Extract links
        right_link = right_item['link']
        left_link = left_item['link']

        # 5. & 6.  Use input titles
        # (already have them as function parameters)

        # 7. notePath (always empty in this example)
        note_path = ""

        # 8. backlink and nextlink
        backlink = False if i == 0 else {
            "text": list_nav[i - 1]['text'],  # Text from the *previous* item in list_a
            "link": f"{url_path}{pre_slug}{os.path.basename(list_nav[i-1]['link']).replace('.md', '')[3:]}" # link from previous item
        }


        nextlink = False if i == min(count, len(list_left), len(list_right)) - 1 else {
            "text": list_nav[i + 1]['text'],  # Text from the *next* item in list_a
            "link": f"{url_path}{pre_slug}{os.path.basename(list_nav[i+1]['link']).replace('.md', '')[3:]}"# link from previous item

        }

        merged_item = {
            "params": {
                "slug": slug,
                "data": {
                    "title": title,
                    "left": left_link,
                    "right": right_link,
                    "leftTitle": left_title,
                    "rightTitle": right_title,
                    "notePath": note_path,
                    "backlink": backlink,
                    "nextlink": nextlink
                }
            }
        }

        merged_list.append(merged_item)

    # Create the output directory if it doesn't exist
    # output_dir = os.path.dirname(output_file)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Write the merged list to the JavaScript file
    tmc_file = output_dir+"tmc.js"
    with open(tmc_file, 'w', encoding='utf-8') as f:
        f.write(f"const {output_val} = ")
        json.dump(merged_list, f, indent=2, ensure_ascii=False)  # Use json.dump
        f.write(";\n")
        f.write(f"export default {output_val};\n")

    print(f"Created tmc.js file at {tmc_file} file")

    # Generate the Markdown table of contents (mucluc.md)
    md_toc = ""
    for item in merged_list:
        md_toc += f'- [{item["params"]["data"]["title"]}]({url_path}{item["params"]["slug"]})\n'

    # Write the Markdown TOC to mucluc.md
    mucluc_path = os.path.join(output_dir, "mucluc.md")
    try:
        with open(mucluc_path, 'w', encoding='utf-8') as outfile:
            outfile.write(md_toc)
        print(f"Successfully generated mucluc.md in {mucluc_path}")
    except Exception as e:
        print(f"Error writing to mucluc.md: {e}")

    return merged_list

import re
import json

def read_filelist(js_file: str) -> list[dict]:
    with open(js_file, 'r', encoding='utf-8') as f:
        content = f.read()

    # Lấy phần nằm trong dấu [ ... ]
    match = re.search(r'\[\s*(.*?)\s*\]', content, re.DOTALL)
    if not match:
        raise ValueError("Không tìm thấy array trong file JS")

    array_str = '[' + match.group(1) + ']'

    # Chuyển JS object keys (không có quotes) sang JSON chuẩn
    # { text: "..." } -> { "text": "..." }
    array_str = re.sub(r'(\{|,)\s*(\w+)\s*:', r'\1 "\2":', array_str)

    # Xóa trailing comma trước ] hoặc }
    array_str = re.sub(r',\s*([}\]])', r'\1', array_str)

    return json.loads(array_str)

# text = "text"
# link = "link"
# count = 500

# left_title = "Pali (vi)"
# right_title = "TK Thích Minh Châu"

# alang = "vi"
# kinh="kinhtuongung"
# author1="pali"
# author2="tmc"
# save_to_dir = f"../docs/{kinh}/c-{author1}-{author2}-{alang}/"  # result file saved directory
# print_author_export = f"{author1}_{author2}_{alang}"  # Example variable name
# print_url_path = f"/{kinh}/c-{author1}-{author2}-{alang}/"
# pre_slug = "snc-"
# pre_title="SNC "
# left_panel = read_filelist('../docs/kinhtuongung/pali-vi/meta/filelist.js')
# right_panel = read_filelist('../docs/kinhtuongung/thichminhchau/meta/filelist.js')

# merged = merge_lists(left_panel, right_panel, count, left_title, right_title, save_to_dir, print_author_export, print_url_path, pre_title=pre_title, pre_slug=pre_slug, title_kind="right")



In [2]:
# step 1 - MAKE COMPARE FILE
# copy list from filelist.js create in make_filelist
text = "text"
link = "link"
# 23
left_panel =  [
 { text: "1. PHẨM RẮN", link: "/kinhtieubo/pali-vi/snp/vagga1-uragavagga.md" },
  { text: "2. CHƯƠNG NHỎ", link: "/kinhtieubo/pali-vi/snp/vagga2-culavagga.md" },
  { text: "3. CHƯƠNG LỚN", link: "/kinhtieubo/pali-vi/snp/vagga3-mahavagga.md" },
  { text: "4. PHẨM TÁM KỆ", link: "/kinhtieubo/pali-vi/snp/vagga4-atthakavagga.md" },
  { text: "5. PHẨM CON ĐƯỜNG ĐẾN BỜ KIA", link: "/kinhtieubo/pali-vi/snp/vagga5-parayanavagga.md" },
]

right_panel = [
  { text: "Chương Một – Phẩm Rắn (Uragavagga)", link: "/docs/kinhtieubo/thichminhchau/kn-018-tap-5-chuong-1-pham-ran-uragavagga.md" },
  { text: "Chương Hai – Tiểu Phẩm", link: "/docs/kinhtieubo/thichminhchau/kn-019-tap-5-chuong-2-tieu-pham.md" },
  { text: "Chương Ba – Ðại Phẩm", link: "/docs/kinhtieubo/thichminhchau/kn-020-tap-5-chuong-3-dai-pham.md" },
  { text: "Chương Bốn – Phẩm Tám", link: "/docs/kinhtieubo/thichminhchau/kn-021-tap-5-chuong-4-pham-tam.md" },
  { text: "Chương Năm – Trên Ðường Ðến Bờ Bên Kia", link: "/docs/kinhtieubo/thichminhchau/kn-022-tap-5-chuong-5-tren-duong-den-bo-ben-kia.md" },
]
count = 500
left_title = "Pali (vi)"
right_title = "HT Thích Minh Châu"

alang = "vi"
kinh="kinhtieubo"
author1="pali"
author2="tmc"
save_to_dir = f"../docs/{kinh}/c-{author1}-{author2}-{alang}/snp/"  # result file saved directory
print_author_export = f"{author1}_{author2}_{alang}"  # Example variable name
print_url_path = f"/{kinh}/c-{author1}-{author2}-{alang}/snp/"
pre_slug = "snp-"
pre_title="snp "
merged = merge_lists(left_panel, right_panel, count, left_title, right_title, save_to_dir, print_author_export, print_url_path, pre_title=pre_title, pre_slug=pre_slug, title_kind="right")
# import json
# print(json.dumps(merged, indent=2, ensure_ascii=False)) # use json for pretty printing

Created tmc.js file at ../docs/kinhtieubo/c-pali-tmc-vi/snp/tmc.js file
Successfully generated mucluc.md in ../docs/kinhtieubo/c-pali-tmc-vi/snp/mucluc.md
